# Setup

In [1]:
import os
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm

The following is run in a linux terminal

`mlst --scheme spyogenes --nopath /mnt/craig/pan_phylon/S_pyogenes/raw/mash_genomes/*.fna > mlst_annotations.txt`

In [2]:
MLST = '/mnt/craig/pan_phylon/S_pyogenes/processed/mlst_annotations.txt'

In [3]:
EMMTYPER = '/mnt/craig/pan_phylon/S_pyogenes/processed/emm_typing_results.tsv'

In [4]:
mlst = pd.read_csv(MLST, sep='\t', header=None, dtype='object')

# Add column names
mlst.columns = ['genome_id', 'schema', 'mlst'] + [allele.split('(')[0] for allele in mlst.iloc[0][3:].values]

mlst

,genome_id,schema,mlst,gki,gtr,murI,mutS,recP,xpt,yqiL
0,1010840.4.fna,spyogenes,172,gki(56),gtr(24),murI(39),mutS(7),recP(30),xpt(2),yqiL(33)
1,1048264.3.fna,spyogenes,36,gki(5),gtr(2),murI(2),mutS(6),recP(6),xpt(2),yqiL(2)
2,1150773.3.fna,spyogenes,37,gki(5),gtr(2),murI(5),mutS(5),recP(5),xpt(4),yqiL(3)
3,1150773.4.fna,spyogenes,37,gki(5),gtr(2),murI(5),mutS(5),recP(5),xpt(4),yqiL(3)
4,1235829.3.fna,spyogenes,28,gki(4),gtr(3),murI(4),mutS(4),recP(4),xpt(2),yqiL(4)
...,...,...,...,...,...,...,...,...,...,...
397,GCF_055378755.1.fna,spyogenes,15,gki(2),gtr(6),murI(8),mutS(5),recP(2),xpt(3),yqiL(2)
398,GCF_900608505.1.fna,spyogenes,28,gki(4),gtr(3),murI(4),mutS(4),recP(4),xpt(2),yqiL(4)
399,GCF_900636425.1.fna,spyogenes,538,gki(11),gtr(2),murI(69),mutS(3),recP(84),xpt(13),yqiL(83)
400,GCF_900636485.1.fna,spyogenes,1271,gki(71),gtr(21),murI(1),mutS(1),recP(5),xpt(3),yqiL(3)


In [5]:
df_emmtyper = pd.read_csv(EMMTYPER, sep='\t', header=None, dtype='object')

# add column names
df_emmtyper.columns = ['genome_id', 'Num of emm-clusters', 'Predicted emm-type', 'Possible emm-like alleles', 'emm cluster']

# Change mangled genome_ids
fna_dir = '/mnt/craig/pan_phylon/S_pyogenes/raw/mash_genomes/'
fna_files = sorted([f.replace('.fna', '') for f in os.listdir(fna_dir) if f.endswith('.fna')])

df_emmtyper['genome_id'] = fna_files

df_emmtyper

,genome_id,Num of emm-clusters,Predicted emm-type,Possible emm-like alleles,emm cluster
0,1010840.4,2,_emm_59.0,_emm_174.1*,E6
1,1048264.3,1,_emm_12.0,NaN,A-C4
2,1150773.3,1,_emm_6.52,NaN,Single protein cluster clade Y
3,1150773.4,1,_emm_6.52,NaN,Single protein cluster clade Y
4,1235829.3,1,_emm_1.0,NaN,A-C3
...,...,...,...,...,...
397,GCF_055378755.1,1,_emm_3.0,NaN,A-C5
398,GCF_900608505.1,1,_emm_1.0,NaN,A-C3
399,GCF_900636425.1,3,_emm_80.0,_emm_156.1~*;_emm_138.1~*,D4
400,GCF_900636485.1,1,_emm_47.1,NaN,Single protein cluster clade Y


# Enrich metadata

## MLST enrichment

In [6]:
cds_metadata = pd.read_csv('/mnt/craig/pan_phylon/S_pyogenes/metadata/cds_metadata.csv', dtype='object', index_col=0)
cds_metadata

,genome_id,source,genome_name,organism_name,taxon_id,strain,# ambiguous bases,# genomes,0,1,...,taxon_lineage_names,temperature_range,total_genes,type_strain,mash_cluster_initial,mash_cluster_filtered,iso_raw,host_cat,source_cat,country_merged
0,1010840.4,BV-BRC,Streptococcus pyogenes MGAS1882,Streptococcus pyogenes MGAS1882,1010840,MGAS1882,NaN,NaN,NaN,NaN,...,cellular organisms;Bacteria;Terrabacteria grou...,NaN,NaN,NaN,89,NaN,NaN,Not Specified,Not Specified,USA
1,1048264.3,BV-BRC,Streptococcus pyogenes HKU QMH11M0907901,NaN,1048264,HKU QMH11M0907901,NaN,NaN,NaN,NaN,...,cellular organisms;Bacteria;Terrabacteria grou...,NaN,NaN,NaN,6,NaN,blood from scarlet fever outbreak,Human,Blood,China
2,1150773.3,BV-BRC,Streptococcus pyogenes JRS4,NaN,1150773,JRS4,NaN,NaN,NaN,NaN,...,cellular organisms;Bacteria;Terrabacteria grou...,NaN,NaN,NaN,7,8.0,NaN,Not Specified,Not Specified,NaN
3,1150773.4,BV-BRC,Streptococcus pyogenes JRS4,NaN,1150773,JRS4,NaN,NaN,NaN,NaN,...,cellular organisms;Bacteria;Terrabacteria grou...,NaN,NaN,NaN,7,8.0,NaN,Not Specified,Not Specified,NaN
4,1235829.3,BV-BRC,Streptococcus pyogenes A20,Streptococcus pyogenes A20,1235829,A20,NaN,NaN,NaN,NaN,...,cellular organisms;Bacteria;Terrabacteria grou...,NaN,NaN,NaN,2,2.0,necrotizing fasciitis patient,Human,Soft Tissue,Taiwan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397,GCF_055378755.1,NCBI,NaN,Streptococcus pyogenes,1314,D58,0.0,30.0,2.0,674.0,...,NaN,NaN,1847.0,NaN,16,6.0,NaN,Not Specified,Not Specified,USA
398,GCF_900608505.1,NCBI,NaN,Streptococcus pyogenes,1314,S119,0.0,240.0,1.0,523.0,...,NaN,NaN,1900.0,NaN,2,2.0,blood culture,Human,Blood,France
399,GCF_900636425.1,NCBI,NaN,Streptococcus pyogenes,1314,NCTC12060,0.0,30.0,4.0,672.0,...,NaN,NaN,1813.0,NaN,27,NaN,not available: to be reported later,Not Specified,Not Specified,United Kingdom
400,GCF_900636485.1,NCBI,NaN,Streptococcus pyogenes,1314,NCTC8232,0.0,30.0,2.0,673.0,...,NaN,NaN,1970.0,NaN,12,NaN,not available: to be reported later,Not Specified,Not Specified,United Kingdom


In [7]:
cds_metadata['mlst'] = None

for idx in tqdm(cds_metadata.index):
    genome_id = cds_metadata.loc[idx, 'genome_id']
    mlst_value = mlst.set_index('genome_id').loc[f'{genome_id}.fna', 'mlst']

    # if non-exact mlst allele match, set to -1
    if mlst_value == '-':
        mlst_value = -1
    
    cds_metadata.loc[idx, 'mlst'] = mlst_value

display(
    cds_metadata.shape,
    cds_metadata.head()
)

  0%|          | 0/402 [00:00<?, ?it/s]

(402, 132)

,genome_id,source,genome_name,organism_name,taxon_id,strain,# ambiguous bases,# genomes,0,1,...,taxon_lineage_names,temperature_range,total_genes,type_strain,mash_cluster_initial,mash_cluster_filtered,iso_raw,host_cat,source_cat,country_merged
0,1010840.4,BV-BRC,Streptococcus pyogenes MGAS1882,Streptococcus pyogenes MGAS1882,1010840,MGAS1882,NaN,NaN,NaN,NaN,...,cellular organisms;Bacteria;Terrabacteria grou...,NaN,NaN,NaN,89,NaN,NaN,Not Specified,Not Specified,USA
1,1048264.3,BV-BRC,Streptococcus pyogenes HKU QMH11M0907901,NaN,1048264,HKU QMH11M0907901,NaN,NaN,NaN,NaN,...,cellular organisms;Bacteria;Terrabacteria grou...,NaN,NaN,NaN,6,NaN,blood from scarlet fever outbreak,Human,Blood,China
2,1150773.3,BV-BRC,Streptococcus pyogenes JRS4,NaN,1150773,JRS4,NaN,NaN,NaN,NaN,...,cellular organisms;Bacteria;Terrabacteria grou...,NaN,NaN,NaN,7,8.0,NaN,Not Specified,Not Specified,NaN
3,1150773.4,BV-BRC,Streptococcus pyogenes JRS4,NaN,1150773,JRS4,NaN,NaN,NaN,NaN,...,cellular organisms;Bacteria;Terrabacteria grou...,NaN,NaN,NaN,7,8.0,NaN,Not Specified,Not Specified,NaN
4,1235829.3,BV-BRC,Streptococcus pyogenes A20,Streptococcus pyogenes A20,1235829,A20,NaN,NaN,NaN,NaN,...,cellular organisms;Bacteria;Terrabacteria grou...,NaN,NaN,NaN,2,2.0,necrotizing fasciitis patient,Human,Soft Tissue,Taiwan


In [8]:
cds_metadata.mlst

0       172
1        36
2        37
3        37
4        28
       ... 
397      15
398      28
399     538
400    1271
401      37
Name: mlst, Length: 402, dtype: object

## emmtyper enrichment

In [9]:
df_emmtyper['Predicted emm-type'] = df_emmtyper['Predicted emm-type'].str.lstrip('_')
cds_metadata = cds_metadata.merge(df_emmtyper, on='genome_id', how='left')

cds_metadata

,genome_id,source,genome_name,organism_name,taxon_id,strain,# ambiguous bases,# genomes,0,1,...,mash_cluster_initial,mash_cluster_filtered,iso_raw,host_cat,source_cat,country_merged,Num of emm-clusters,Predicted emm-type,Possible emm-like alleles,emm cluster
0,1010840.4,BV-BRC,Streptococcus pyogenes MGAS1882,Streptococcus pyogenes MGAS1882,1010840,MGAS1882,NaN,NaN,NaN,NaN,...,89,NaN,NaN,Not Specified,Not Specified,USA,2,emm_59.0,_emm_174.1*,E6
1,1048264.3,BV-BRC,Streptococcus pyogenes HKU QMH11M0907901,NaN,1048264,HKU QMH11M0907901,NaN,NaN,NaN,NaN,...,6,NaN,blood from scarlet fever outbreak,Human,Blood,China,1,emm_12.0,NaN,A-C4
2,1150773.3,BV-BRC,Streptococcus pyogenes JRS4,NaN,1150773,JRS4,NaN,NaN,NaN,NaN,...,7,8.0,NaN,Not Specified,Not Specified,NaN,1,emm_6.52,NaN,Single protein cluster clade Y
3,1150773.4,BV-BRC,Streptococcus pyogenes JRS4,NaN,1150773,JRS4,NaN,NaN,NaN,NaN,...,7,8.0,NaN,Not Specified,Not Specified,NaN,1,emm_6.52,NaN,Single protein cluster clade Y
4,1235829.3,BV-BRC,Streptococcus pyogenes A20,Streptococcus pyogenes A20,1235829,A20,NaN,NaN,NaN,NaN,...,2,2.0,necrotizing fasciitis patient,Human,Soft Tissue,Taiwan,1,emm_1.0,NaN,A-C3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397,GCF_055378755.1,NCBI,NaN,Streptococcus pyogenes,1314,D58,0.0,30.0,2.0,674.0,...,16,6.0,NaN,Not Specified,Not Specified,USA,1,emm_3.0,NaN,A-C5
398,GCF_900608505.1,NCBI,NaN,Streptococcus pyogenes,1314,S119,0.0,240.0,1.0,523.0,...,2,2.0,blood culture,Human,Blood,France,1,emm_1.0,NaN,A-C3
399,GCF_900636425.1,NCBI,NaN,Streptococcus pyogenes,1314,NCTC12060,0.0,30.0,4.0,672.0,...,27,NaN,not available: to be reported later,Not Specified,Not Specified,United Kingdom,3,emm_80.0,_emm_156.1~*;_emm_138.1~*,D4
400,GCF_900636485.1,NCBI,NaN,Streptococcus pyogenes,1314,NCTC8232,0.0,30.0,2.0,673.0,...,12,NaN,not available: to be reported later,Not Specified,Not Specified,United Kingdom,1,emm_47.1,NaN,Single protein cluster clade Y


# Save enriched metadata file

In [10]:
cds_metadata.to_csv('/mnt/craig/pan_phylon/S_pyogenes/metadata/enriched_metadata.csv')